
## **Laboratorium 4. Fabryka konfiguracji**

W poprzednich laboratoriach konfiguracja była wczytywana z pliku YAML, iterowana, spłaszczana i mapowana na obiekty dataclass. W tym laboratorium rozwijamy ten mechanizm w kierunku programowania obiektowego: wydzielamy odpowiedzialności do klas, wprowadzamy fabrykę konfiguracji, a także przygotowujemy kod tak, aby był łatwy do rozbudowy o nowe typy sekcji i nowe źródła konfiguracji.

## Poziom podstawowy – Obiektowa reprezentacja sekcji konfiguracji

# **Założenia**

Program powinien:

* wczytywać konfigurację YAML,
* parsować wybrane sekcje do obiektów,
* wykorzystywać klasy reprezentujące poszczególne sekcje konfiguracji,
* spełniać implementacje niżej wymienonych metod.

# **Wymagania**

Utwórz data-klasy reprezentujące **wszystkie** sekcje konfiguracji:

* `AppConfig`
* `ServerConfig`
* `DatabaseConfig`

(Na ten moment proszę odłożyć na bok klasę do zebrania w.w. klas)

Każda klasa powinna:

* przechowywać dane sekcji,
* posiadać metodę validate() ewentualnie rzucającą wyjątek (nic nie zwraca),
* posiadać implementację metody __str__() dla ustawienia formatowania `str(dataobiekt)`
* posiadać metodę display() do wyświetlenia zawartośći klasy (nic nie zwraca).


In [ ]:
# Przykładowy szkic, uwzgledniający abstrakcyjną klasę bazową (będzie potrzebna lada moment!)
from dataclasses import dataclass, fields

class BaseConfigSection:
    pass

@dataclass(slots=True, frozen=True)
class AppConfig(BaseConfigSection):
    name: str
    debug: bool

    def validate(self) -> None:
        if not isinstance(self.name, str):
            raise TypeError("Pole 'name' musi być typu str.")
        if not isinstance(self.debug, bool):
            raise TypeError("Pole 'debug' musi być typu bool.")

        if not self.name:
            raise ValueError("Pole 'name' nie może być puste.")
        if len(self.name) < 3:
            raise ValueError("Pole 'name' musi zawierać conajmniej 3 litery.")
        if not self.name.isalpha():
            raise ValueError("Pole 'name' możę zawierać tylko litery.")

    def __str__(self) -> str:
        return ", ".join(
            f"{field.name}={getattr(self, field.name)!r}"
            for field in fields(self)
        )

    def display(self) -> None:
        print(f"AppConfig: {str(self)}")

# **Efekt końcowy**

Program powinien:

* utworzyć obiekty dla wszystkich sekcji konfiguracji,
* wywołać ich metody walidujące,
* wyświetlić dane w czytelnej formie.

Przykład:

```python
app = AppConfig("MojaAplikacja", True)
app.validate()
app.display()
```



# **Poziom średniozaawansowany – Fabryka konfiguracji**

Wprowadź wzorzec projektowy [Factory](https://refactoring.guru/design-patterns/factory-method) ([przykład](https://refactoring.guru/design-patterns/factory-method/python/example)), który będzie odpowiadał za tworzenie odpowiednich obiektów konfiguracji na podstawie danych wejściowych.

# Założenia

Program powinien realizować wszystko z poziomu łatwego, a dodatkowo:

* zawierać klasę `ConfigFactory`,
* fabryka powinna tworzyć odpowiedni obiekt sekcji na podstawie nazwy sekcji,
* logika tworzenia obiektów nie powinna znajdować się w kodzie głównym.

# Wymagania
Utwórz klasę:


```python
from enum import StrEnum

class SectionType(StrEnum):
    APP = "app"
    SERVER = "server"
    DATABASE = "database"

class ConfigFactory:

    @staticmethod
    def create_section(section_name: str, data: dict) -> BaseConfig:
        section = SectionType(section_name.lower())
        match section:
            case SectionType.APP:
                return AppConfig(**data)
            case SectionType.SERVER:
                return # TODO ...
            case SectionType.DATABASE:
                return # TODO ...
            case _:
                raise # TODO ...
```

# Fabryka powinna:

* dla "app" zwrócić obiekt AppConfig,
* dla "server" zwrócić obiekt ServerConfig,
* dla "database" zwrócić obiekt DatabaseConfig,
* dla nieznanej sekcji zgłosić wyjątek, np. ValueError.



## Przykład użycia:

```python
loaded_server_yaml = {
    "host": "127.0.0.1",
    "port": 8080,
    "timeout": 30
}
section_obj = ConfigFactory.create_section("server", loaded_server_yaml)
section_obj.display()
```

# Dodatkowe założenie

Program powinien iterować po wszystkich sekcjach YAML i dla każdej sekcji wywoływać metodę fabryki.

In [ ]:
# Przykład:
objects = {}
for section_name, section_data in config.items():
    obj = ConfigFactory.create_section(section_name, section_data["App"])
    obj.validate()
    objects[section_name] = obj

print(str(objects))

# **Efekt końcowy**

Program powinien budować słownik gotowych obiektów konfiguracyjnych bez ręcznego if/else w kodzie głównym.

# **Poziom zaawansowany – Hierarchia klas i wspólny interfejs sekcji**

Zaprojektuj wspólną architekturę obiektową dla wszystkich sekcji konfiguracji, wykorzystując dziedziczenie lub klasy abstrakcyjne.

*(W tym miejscu wracamy do klasy zbiorczej, dla wszystkich dataklas)*

# Założenia

Program powinien realizować wszystko z poprzednich poziomów, a dodatkowo:

* zawierać klasę bazową dla sekcji konfiguracji (wyciągniecie wspólnej funkcjonalności),
* wymuszać wspólny interfejs dla wszystkich typów sekcji,
* umożliwiać łatwe dodanie nowych sekcji bez modyfikowania kodu głównego.

# Wymagania

W dotychczas pustej klasie abstrakcyjnej wprowadź następujące zmiany:
1. Dziedzicz po klasie `ABC`
2. Określ interfejs (listę metod), które muszą zostać zaimplementowane w każdej klasie dziedziczącej przy pomocy dekoratorów `@abstractmethod`

Przykład:

```python
from abc import ABC, abstractmethod

class BaseConfigSection(ABC):
    @abstractmethod
    def validate(self):
        pass

    @abstractmethod
    def display(self):
        pass
```

# Następnie:

* AppConfig, ServerConfig, DatabaseConfig powinny dziedziczyć po BaseConfigSection,
* każda klasa musi implementować validate() i display().

# Dodatkowo utwórz klasę nadrzędną, np. ApplicationConfig, która:

* przechowuje wszystkie sekcje jako obiekty,
* umożliwia wyświetlenie pełnej konfiguracji,
* posiada metodę validate_all().

In [ ]:
# Przykład:

# Umozliwenie edycji calych sekcji
@dataclass(slots=True, frozen=False)
class ApplicationConfig:
    sections: dict[str, BaseConfigSection]

    def validate_all(self):
        for section in self.sections.values():
            section.validate()

    def display_all(self):
        for name, section in self.sections.items():
            print(f"[{name}] {str(section)}")

# Efekt końcowy

Po uruchomieniu program powinien:

* utworzyć pełny obiekt konfiguracji aplikacji,
* zwalidować wszystkie sekcje,
* wypisać całą konfigurację,
* działać zgodnie z zasadami programowania obiektowego: hermetyzacja, podział odpowiedzialności, rozszerzalność.

# **Zadanie dodatkowe – Rejestr fabryk i dynamiczne rozszerzanie systemu**

Rozszerz fabrykę tak, aby można było dynamicznie rejestrować nowe typy sekcji bez modyfikowania metody create_section().

# Założenia

Zamiast pisać wiele instrukcji 'if-elif' / 'match-case', stwórz mechanizm rejestracji klas w fabryce.

# Wymagania

Fabryka powinna posiadać rejestr klas:

In [ ]:
class ConfigFactory:
    _registry = {}

    @classmethod
    def register(cls, section_name: str, section_class):
        cls._registry[section_name] = section_class

    @classmethod
    def create_section(cls, section_name: str, data: dict):
        if section_name not in cls._registry:
            raise ValueError(f"Nieznana sekcja: {section_name}")
        return cls._registry[section_name](**data)

In [ ]:
# Przykład użycia:
ConfigFactory.register("app", AppConfig)
ConfigFactory.register("server", ServerConfig)
ConfigFactory.register("database", DatabaseConfig)

Następnie dodaj nową sekcję, np. logging, bez zmiany kodu samej fabryki:

In [ ]:
class LoggingConfig(BaseConfigSection):
    ...
ConfigFactory.register("logging", LoggingConfig)

# Efekt końcowy

System powinien dawać możliwość:

* łatwego dodawania nowych sekcji,
* zachowania zgodności z OOP,
* unikania rozbudowanych instrukcji warunkowych.